# Fine Tuning A Chatbot for Medical questions:
Developing a medicl chat system using medicl Q and A data.

## Part 3:
Using and Evaluating our model






In [6]:
# Install the packages we may need
!pip install -q transformers datasets accelerate bitsandbytes peft trl torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 48.9 MB/s eta 0:00:00


In [55]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [11]:
# confirm we have cuda
import torch
print(torch.cuda.is_available())

True


In [16]:
# upload previously created model files
import shutil

# Path to the ZIP file
zip_file_path = "/llama-lora-finetuned.zip"


# Directory where the contents will be extracted
destination_directory = "llama-finetuned-medchat-3"

shutil.unpack_archive(zip_file_path, destination_directory, "zip")

In [2]:
TOKEN=<redacted>

### 3a: Create a handler class
Useful to run our model, and will also help if we wan to use a hugging face inference endpoint.

In [37]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
import torch

# Much of this code borrowed from huggingface docs.
# See https://huggingface.co/docs/inference-endpoints/main/en/engines/toolkit#create-a-custom-inference-handler.


class EndpointHandler:
    def __init__(self, path=""):
        """
        path: directory where your adapter is stored
        """
        base_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=TOKEN)

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            token=TOKEN
        )

        # Load LoRA adapter
        self.model = PeftModel.from_pretrained(base_model, path)

        # Build text-generation pipeline
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )
        self.device = 0 if torch.cuda.is_available() else -1


    def __call__(self, data):
        """
        data: dict from the inference request
        """
        inputs = data.get("inputs", "")
        parameters = data.get("parameters", {}) or {}

        # format input to our expected chat formal
        messages = [
            {"role": "user", "content": inputs}
        ]
        # match the template used in training
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # tokenize prompt
        model_inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # generate output
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=parameters.get("max_new_tokens", 256),
            temperature=parameters.get("temperature", 0.7),
            top_p=parameters.get("top_p", 0.9),
            do_sample=parameters.get("do_sample", True)
        )

        # decode response
        decoded = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        # Strip prompt to leave only assistant answer
        # this is not consistently working for me
        #assistant_response = decoded[len(prompt):].strip()
        assistant_response = decoded.strip()
        return {"generated_text": assistant_response}



### 3b: Run the model
Example model interactions.

In [33]:
data = {
    "inputs": "What is glaucoma.",
}
my_handler = EndpointHandler(path="llama-finetuned-medchat-3")
output=my_handler(data)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [25]:
print(output['generated_text'])

What is glaucoma.?
Glaucoma is a condition that damages the optic nerve, the health of which is vital for good vision. This damage is often caused by abnormally high pressure in the eye. It can lead to blindness if not treated. It usually affects both eyes. The damage to the optic nerve is usually permanent and cannot be reversed. Early treatment can slow or stop disease progression and reduce the risk of vision loss. If left untreated, glaucoma can cause blindness. It is one of the leading causes


In [35]:

data = {
    "inputs": "What is cancer?",
}
print(my_handler(data)["generated_text"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


and spread to other parts of the body. Cancer can begin in almost any part of the body, including the breast, lung, colon, prostate, stomach, skin, uterus, and brain. There are over 100 kinds of cancer. Cancer cells are abnormal cells with changes (mutations) in their DNA. The cells do not die as they normally do and new cells grow to replace them. As the number of abnormal cells grows, they form a mass called a tumor. Tumors can be benign or malignant. Benign tumors are not cancer and do not spread to other parts of the body. Malignant tumors are cancer and can spread.


In [36]:
data = {
    "inputs": "How can I treat high blood pressure?",
    "parameters": {"max_new_tokens": 100}
}
output=my_handler(data)

print(output['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


and low-fat dairy products. The DASH diet is also low in fat, red meat, sweets, and salt. You may also be advised to lose weight if you're overweight or obese. You should also try to get at least 30 minutes of moderate physical activity each day. Exercise, such as walking, can lower your blood pressure and improve your


In [27]:
data = {
    "inputs": "What is fibromyalgia?",
    "parameters": {"max_new_tokens": 100}
}
output=my_handler(data)

print(output['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What is fibromyalgia? Fibromyalgia is a common and chronic condition characterized by widespread musculoskeletal pain accompanied by fatigue, sleep, memory, and mood problems. The condition is also often associated with gastrointestinal issues, headaches, and sensitivity to light and sound. Fibromyalgia is a long-term condition, and it can be challenging to diagnose. Symptoms vary from person to person, but common symptoms include: Pain in the muscles and joints, usually on both sides of the body, that is described as deep, a


In [39]:
data = {
    "inputs": "What is cancer?",
    "parameters": {"max_new_tokens": 100}
}
my_handler = EndpointHandler(path="llama-finetuned-medchat-3")

output=my_handler(data)

print(output['generated_text'])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What is cancer?assistant

The body is made up of many types of cells. Normally, cells grow, divide, and die in an orderly way. During the life of a cell, it goes through three stages of development: growth, reproduction, and death. Cancer results from an abnormality in one or more genes in the DNA. This results in an uncontrolled growth of abnormal cells.


In [43]:
output['generated_text']

'system\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nuser\n\nWhat is cancer?assistant\n\nThe body is made up of many types of cells. Normally, cells grow, divide, and die in an orderly way. During the life of a cell, it goes through three stages of development: growth, reproduction, and death. Cancer results from an abnormality in one or more genes in the DNA. This results in an uncontrolled growth of abnormal cells.'

In [44]:
output['generated_text'].split("\n")[-1]

'The body is made up of many types of cells. Normally, cells grow, divide, and die in an orderly way. During the life of a cell, it goes through three stages of development: growth, reproduction, and death. Cancer results from an abnormality in one or more genes in the DNA. This results in an uncontrolled growth of abnormal cells.'

In [45]:
data = {
    "inputs": "Is cancer contagious?",
}
my_handler = EndpointHandler(path="llama-finetuned-medchat-3")

output=my_handler(data)

print(output['generated_text'])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Is cancer contagious?assistant

Yes, some cancers are contagious.  The most common contagious cancer is a type of skin cancer that affects dogs.  A contagious form of skin cancer also affects Tasmanian devils.


In [46]:
output['generated_text'].split("\n")[-1]

'Yes, some cancers are contagious.  The most common contagious cancer is a type of skin cancer that affects dogs.  A contagious form of skin cancer also affects Tasmanian devils.'

In [48]:
# how will the model respond to unexpected input?
data = {
    "inputs": "How are you?",
}
#my_handler = EndpointHandler(path="llama-finetuned-medchat-3")

output=my_handler(data)

print(output['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

How are you?assistant

How are you? I'm functioning well. Is there something specific you'd like to know about me?


In [49]:
# Will the model make up an answer for a fake disease?

data = {
    "inputs": "What is sarahgc disease?",
}

output=my_handler(data)

print(output['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What is sarahgc disease?assistant

Sarahgc disease is a condition that affects the nervous system. People with this condition have muscle weakness, primarily in the arms and legs. The muscles of the face, throat, and tongue are also affected, leading to difficulties with swallowing and speech. This condition also causes impaired vision. It is caused by a mutation in the TULP1 gene and is inherited in an autosomal recessive fashion.


### 3c: Thoughts
I don't like that the model is making up answers to fake diseases. Maybe we can add in some sort of system prompt after the fact to tell it to respond "I don't know" if it doesn't know something.

It also seems I will need to do some tweaking to clean the output text. I will move forward with evalutation but plan to improve this later.

### 3d: Evaluation

In [52]:
# I  need to split my dataset again because I lost my data with the runtime ending in collab.
# Luckily our seed and test_size can be used again to create the same split
from datasets import load_dataset

dataset = load_dataset("json", data_files="out.jsonl")

# split train/test (90/10)
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
eval_dataset = [{
        "question": ex["input"],
        "context": "",
        "answers": [ex["output"]]
    }
    for ex in dataset["test"]
]

In [65]:
results = task_evaluator.compute(
    model_or_pipeline="llama-finetuned-medchat-3",
    data=dataset["test"],
    metric="squad"
)

print(results)

ValueError: Invalid `question_column` question specified. The dataset contains the following columns: ['text'].

Unfortunatley I have hit a limit in my google collab instance. I need to reformat my data if I want to use the question-answer evaluation from hugging face. I will come back to this later.